# Question
### Do teams that advanced past the group stage have a significantly higher average possession percentage than teams eliminated in the group stage?

## Step 1 — Load raw match data and Round-of-32 reference list
- `raw_data.csv` holds only what was **manually collected** — team, match, and possession % per match.
- `round_of_32_teams.csv` holds the 32 team names that reached the Round of 32.


In [12]:
import pandas as pd

In [13]:
# raw match data
match_df = pd.read_csv("raw_data.csv")
print(f"Loaded {len(match_df)} rows from raw_data.csv")
match_df.head()

Loaded 144 rows from raw_data.csv


,team,match_id,possession_pct
0,Mexico,Mexico_vs_SouthAfrica_2026-06-11,61
1,South Africa,Mexico_vs_SouthAfrica_2026-06-11,40
2,Korea Republic,KoreaRepublic_vs_Czechia_2026-06-11,62
3,Czechia,KoreaRepublic_vs_Czechia_2026-06-11,38
4,Canada,Canada_vs_BosniaHerz2026-06-12,61


In [14]:
# round of 32 reference list
round_of_32_df = pd.read_csv("round_of_32_teams.csv")
print(f"Loaded {len(round_of_32_df)} teams from round_of_32_teams.csv")
round_of_32_df.head()

Loaded 32 teams from round_of_32_teams.csv


,team
0,South Africa
1,Canada
2,Brazil
3,Japan
4,Germany


## Step 2 — Data cleaning and wrangling
1. **Clean team names** — strip unwanted leading/trailing whitespace. This matters: `"Algeria"` and `"Algeria "` are treated as two different strings by pandas, which silently breaks any merge based on exact name matching.
2. **Merge** `match_df` with `round_of_32_df` to attach an `advanced` flag (1 = reached Round of 32, 0 = eliminated in the group stage)

In [15]:
# Clean whitespace from team names in both files
before = match_df["team"].copy()
match_df["team"] = match_df["team"].astype(str).str.strip()
n_cleaned = (before != match_df["team"]).sum()
print(f"Cleaned whitespace from 'team' in {n_cleaned} row(s) of raw_data.csv")

round_of_32_df["team"] = round_of_32_df["team"].astype(str).str.strip()

Cleaned whitespace from 'team' in 71 row(s) of raw_data.csv


In [16]:
# Attach the advanced/eliminated flag via merge
round_of_32_df = round_of_32_df.copy()
round_of_32_df["advanced"] = 1

df = match_df.merge(round_of_32_df, on="team", how="left")
df["advanced"] = df["advanced"].fillna(0).astype(int)

# Sanity check: any Round-of-32 team not found in the match data at all?
unmatched = set(round_of_32_df["team"]) - set(match_df["team"])
if unmatched:
    print(f"WARNING: these Round-of-32 teams were not found in raw_data.csv: {unmatched}")
else:
    print("All 32 Round-of-32 team names matched successfully.")

All 32 Round-of-32 team names matched successfully.


In [17]:
# Basic validation
df = df.dropna(subset=["possession_pct"])
df["possession_pct"] = df["possession_pct"].astype(float)

n_advanced = (df["advanced"] == 1).sum()
n_eliminated = (df["advanced"] == 0).sum()
print(f"Population sizes -> advanced: {n_advanced}, eliminated: {n_eliminated}")
if n_advanced != 96 or n_eliminated != 48:
    print("NOTE: population sizes differ from the expected 96 / 48")
    
df.head()

Population sizes -> advanced: 96, eliminated: 48


,team,match_id,possession_pct,advanced
0,Mexico,Mexico_vs_SouthAfrica_2026-06-11,61.0,1
1,South Africa,Mexico_vs_SouthAfrica_2026-06-11,40.0,1
2,Korea Republic,KoreaRepublic_vs_Czechia_2026-06-11,62.0,0
3,Czechia,KoreaRepublic_vs_Czechia_2026-06-11,38.0,0
4,Canada,Canada_vs_BosniaHerz2026-06-12,61.0,1


## Step 3 — Data preparation and sampling

Population = 96 advanced team-match records, 48 eliminated team-match records.

A simple random sample of **n = 30** is drawn from each group, without replacement, using a fixed random seed for reproducibility.

In [18]:
RANDOM_SEED = 42
SAMPLE_SIZE_PER_GROUP = 30

advanced_pop = df[df["advanced"] == 1]
eliminated_pop = df[df["advanced"] == 0]

advanced_sample = advanced_pop.sample(n=SAMPLE_SIZE_PER_GROUP, random_state=RANDOM_SEED, replace=False)
eliminated_sample = eliminated_pop.sample(n=SAMPLE_SIZE_PER_GROUP, random_state=RANDOM_SEED, replace=False)

print(f"Advanced sample: n={len(advanced_sample)}")
print(f"Eliminated sample: n={len(eliminated_sample)}")

Advanced sample: n=30
Eliminated sample: n=30
